# strict_retention_three_fold_handoff_v1

Notebook-first model-selection handoff for the current SOXL retention objective.

Purpose: combine the strict walk-forward retention gate, the qlib-aligned SOXLBacktest buy/sell reference, and the three-fold BC/PPO checkpoint selector into one no-promotion training handoff.

Acceptance target used here: all required walk-forward windows present, worst walk-forward MaxDD below 20%, and return aligned to the current best model requirement.

## Inputs

- Strict model retention gate: `runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json`
- SOXLBacktest qlib reference baseline: `runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json`
- Three-fold BC slice readiness: `runtime/model_selection/card/cap10_three_fold_bc_slice_screen_v1/selection.json`
- Three-fold checkpoint drift selector: `runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1/selection.json`
- Checkpoint fold rows: `checkpoint_selection_by_fold.csv`
- Checkpoint candidate rows: `checkpoint_candidate_metrics.csv`

In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'AGENTS.md').exists():
    ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'AGENTS.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.strict_retention_handoff import (
    derive_strict_retention_handoff,
    render_strict_retention_handoff,
)

CARD = ROOT / 'runtime/model_selection/card/strict_retention_three_fold_handoff_v1'
PATHS = {
    'strict_gate_summary': ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json',
    'soxlbacktest_qlib_summary': ROOT / 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json',
    'bc_slice_selection': ROOT / 'runtime/model_selection/card/cap10_three_fold_bc_slice_screen_v1/selection.json',
    'checkpoint_selection': ROOT / 'runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1/selection.json',
    'checkpoint_selection_by_fold': ROOT / 'runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1/checkpoint_selection_by_fold.csv',
    'checkpoint_candidate_metrics': ROOT / 'runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1/checkpoint_candidate_metrics.csv',
}

def repo_path(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

for name, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f'{name}: {path}')

strict_payload = json.loads(PATHS['strict_gate_summary'].read_text(encoding='utf-8'))
strict_summary = strict_payload['summary']
soxlbacktest_summary = json.loads(PATHS['soxlbacktest_qlib_summary'].read_text(encoding='utf-8'))
bc_selection = json.loads(PATHS['bc_slice_selection'].read_text(encoding='utf-8'))
checkpoint_selection = json.loads(PATHS['checkpoint_selection'].read_text(encoding='utf-8'))
selection_by_fold = pd.read_csv(PATHS['checkpoint_selection_by_fold'])
candidate_metrics = pd.read_csv(PATHS['checkpoint_candidate_metrics'])

source_manifest = pd.DataFrame([
    {'name': name, 'path': repo_path(path), 'exists': path.exists()} for name, path in PATHS.items()
])
display(source_manifest)

,name,path,exists
0,strict_gate_summary,runtime/model_selection/card/model_retention_w...,True
1,soxlbacktest_qlib_summary,runtime/model_selection/card/soxlbacktest_qlib...,True
2,bc_slice_selection,runtime/model_selection/card/cap10_three_fold_...,True
3,checkpoint_selection,runtime/model_selection/card/cap10_three_fold_...,True
4,checkpoint_selection_by_fold,runtime/model_selection/card/cap10_three_fold_...,True
5,checkpoint_candidate_metrics,runtime/model_selection/card/cap10_three_fold_...,True


In [2]:
handoff = derive_strict_retention_handoff(
    strict_gate_summary=strict_summary,
    checkpoint_selection_by_fold=selection_by_fold,
    checkpoint_candidate_metrics=candidate_metrics,
)

display(Markdown(render_strict_retention_handoff(handoff)))

# Strict Retention Training Handoff

- decision: `train_bull_trend_participation_with_drawdown_drift_stop_no_promotion`
- promotion: `no promotion`
- actor_retention_allowed: `False`
- ppo_allowed_next: `False`
- reference_baseline: `soxl_backtest_policy_reimplementation_v1`

## Objective Terms

- `soxlbacktest_reference_alignment`
- `walkforward_relative_return_gap_penalty`
- `checkpoint_objective_loss`
- `wf2020_trend_participation_floor`
- `drawdown_drift_penalty`

## Fold Blockers

- wf_2020: `trend_underparticipation`, relative=-285.449375pp, MaxDD=19.417943%

## Stop Conditions

- `do_not_promote_missing_walkforward_windows`
- `do_not_promote_worst_wf_maxdd_ge_20`
- `do_not_promote_ytd_return_below_best_alignment`
- `do_not_promote_without_soxlbacktest_reference_alignment`
- `do_not_continue_same_ppo_when_maxdd_drift_exceeds_2pp`


In [3]:
fold_blockers = pd.DataFrame(handoff.get('fold_blockers', []))
wf2020_candidates = candidate_metrics[candidate_metrics['fold'].astype(str).eq('wf_2020')].copy()

summary = {
    'experiment': 'strict_retention_three_fold_handoff_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': handoff['decision'],
    'promotion_allowed': handoff['promotion_allowed'],
    'actor_retention_allowed': handoff['actor_retention_allowed'],
    'ppo_allowed_next': handoff['ppo_allowed_next'],
    'diagnostic_training_allowed_next': handoff['diagnostic_training_allowed_next'],
    'reference_baseline': handoff['reference_baseline'],
    'reference_return_pct': handoff.get('reference_return_pct'),
    'reference_max_drawdown_pct': handoff.get('reference_max_drawdown_pct'),
    'current_best_return_pct': handoff.get('current_best_return_pct'),
    'required_best_return_pct': handoff.get('required_best_return_pct'),
    'max_walkforward_drawdown_pct': handoff.get('max_walkforward_drawdown_pct'),
    'required_wf_windows': handoff.get('required_wf_windows'),
    'strict_gate_passing_profile_count': strict_summary.get('passing_profile_count'),
    'strict_gate_profiles_scanned': strict_summary.get('profiles_scanned'),
    'soxlbacktest_reference_decision': soxlbacktest_summary.get('decision'),
    'soxlbacktest_selected_buy_sell_reference': soxlbacktest_summary.get('selected_buy_sell_reference'),
    'soxlbacktest_selected_risk_floor': soxlbacktest_summary.get('selected_risk_floor'),
    'soxlbacktest_legacy_default': soxlbacktest_summary.get('legacy_default'),
    'soxlbacktest_wf_pass_ytd18_15_15_02': soxlbacktest_summary.get('wf_pass_ytd18_15_15_02'),
    'bc_slice_decision': bc_selection.get('decision'),
    'checkpoint_selector_decision': checkpoint_selection.get('decision'),
    'checkpoint_selector_failed_folds': checkpoint_selection.get('failed_folds'),
    'primary_failed_fold': handoff.get('primary_failed_fold'),
    'failed_fold_count': handoff.get('failed_fold_count'),
    'objective_terms': handoff.get('objective_terms'),
    'stop_conditions': handoff.get('stop_conditions'),
    'next_action': 'train a bull-trend participation objective with drawdown-drift stop; do not continue the same PPO branch after wf_2020 MaxDD drift',
    'sidecars': {
        'handoff_json': repo_path(CARD / 'strict_retention_handoff.json'),
        'handoff_markdown': repo_path(CARD / 'strict_retention_handoff.md'),
        'fold_blockers': repo_path(CARD / 'fold_blockers.csv'),
        'source_manifest': repo_path(CARD / 'source_manifest.csv'),
        'summary': repo_path(CARD / 'summary.json'),
        'conclusion': repo_path(CARD / 'conclusion.md'),
    },
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
display(fold_blockers)
display(wf2020_candidates[['candidate', 'candidate_kind', 'test_return', 'test_buy_and_hold_return', 'test_relative_return', 'test_max_dd', 'relative_delta_vs_bc', 'max_dd_delta_vs_bc', 'target_gate', 'drift_gate', 'candidate_gate']])

,value
experiment,strict_retention_three_fold_handoff_v1
created_at,2026-06-15T17:49:01.098276+00:00
decision,train_bull_trend_participation_with_drawdown_d...
promotion_allowed,False
actor_retention_allowed,False
ppo_allowed_next,False
diagnostic_training_allowed_next,True
reference_baseline,soxl_backtest_policy_reimplementation_v1
reference_return_pct,145.736343
reference_max_drawdown_pct,13.554089


,fold,blocker,selected_candidate,selected_kind,selected_relative_return_pp,selected_max_dd_pct,selected_target_gate,selected_drift_gate,selected_candidate_gate,ppo_max_dd_drift_pct,ppo_relative_improvement_pp,ppo_drift_candidate,ppo_drift_stop
0,wf_2020,trend_underparticipation,bc_initialization,bc_initialization,-285.449375,19.417943,False,True,False,5.91584,5.981494,ppo_retained_iter_4,True


,candidate,candidate_kind,test_return,test_buy_and_hold_return,test_relative_return,test_max_dd,relative_delta_vs_bc,max_dd_delta_vs_bc,target_gate,drift_gate,candidate_gate
0,bc_initialization,bc_initialization,46.810017,355.73616,-285.449375,19.417943,0.000000,0.00000,False,True,False
1,ppo_retained_iter_0_bc_equivalent,ppo_retained,46.810017,355.73616,-285.449375,19.417943,0.000000,0.00000,False,True,False
2,ppo_retained_iter_4,ppo_retained,41.202940,355.73616,-279.467881,25.333784,5.981494,5.91584,False,False,False
3,ppo_selected_best,ppo_selected,41.202940,355.73616,-279.467881,25.333784,5.981494,5.91584,False,False,False


In [4]:
CARD.mkdir(parents=True, exist_ok=True)
(CARD / 'strict_retention_handoff.json').write_text(json.dumps(handoff, ensure_ascii=True, indent=2) + '\n', encoding='utf-8')
(CARD / 'strict_retention_handoff.md').write_text(render_strict_retention_handoff(handoff), encoding='utf-8')
(CARD / 'summary.json').write_text(json.dumps(summary, ensure_ascii=True, indent=2) + '\n', encoding='utf-8')
source_manifest.to_csv(CARD / 'source_manifest.csv', index=False)
fold_blockers.to_csv(CARD / 'fold_blockers.csv', index=False)

conclusion = f"""# strict_retention_three_fold_handoff_v1 conclusion

Decision: `{handoff['decision']}`. Promotion remains blocked and no actor is retained.

Why: strict retention has `{strict_summary.get('passing_profile_count')}/{strict_summary.get('profiles_scanned')}` passing profiles. Required return is `{strict_summary.get('required_best_return_pct'):.6f}%`, current best return is `{strict_summary.get('current_best_return_pct'):.6f}%`, and every retained model candidate still must prove `{', '.join(strict_summary.get('required_wf_windows') or [])}` with MaxDD below `{strict_summary.get('max_walkforward_drawdown_pct'):.2f}%`.

SOXLBacktest status: `{soxlbacktest_summary.get('decision')}`. It is a qlib-aligned buy/sell reference baseline, not a promotable actor. The selected buy/sell reference is `{soxlbacktest_summary.get('selected_buy_sell_reference')}`; its legacy default has worst WF MaxDD `{soxlbacktest_summary.get('legacy_default', {}).get('worst_wf_max_drawdown_pct')}` and YTD return `{soxlbacktest_summary.get('legacy_default', {}).get('ytd_return_pct')}`.

Three-fold checkpoint status: `{checkpoint_selection.get('decision')}`. Failed folds: `{checkpoint_selection.get('failed_folds')}`. The primary failed fold is `{handoff.get('primary_failed_fold')}`.

Next action: use `{', '.join(handoff.get('objective_terms', []))}` as the next training/checkpoint objective. Do not continue the same PPO branch when MaxDD drift exceeds 2pp.
"""
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print('wrote', CARD)

wrote /Users/rziiiii/dev/workspace/runtime/model_selection/card/strict_retention_three_fold_handoff_v1


## Runtime Conclusion

Decision: `train_bull_trend_participation_with_drawdown_drift_stop_no_promotion`. Promotion is blocked and no actor is retained.

Evidence:

- Strict retention gate has `0/12` passing profiles. The current best YTD return is `224.361539%`; the required best-alignment return is `213.143462%`; required walk-forward windows are `wf_2020`, `wf_2022`, and `wf_2023_2024`; the MaxDD ceiling for this user objective is `20.00%`.
- SOXLBacktest is now a qlib-aligned buy/sell reference baseline, not a retained actor. Its `legacy_default` buy/sell path has YTD return `132.077977%` and worst WF MaxDD `84.918053%`; low-risk SOXLBacktest caps can pass the 20% WF MaxDD line but miss the return gate by a wide margin.
- The three-fold checkpoint selector fails `wf_2020`. The selected BC initialization keeps MaxDD at `19.417943%` but has relative return `-285.449375pp`, which is severe bull-trend underparticipation.
- PPO retained iter 4 in `wf_2020` improves relative return by only `+5.981494pp` while MaxDD drifts by `+5.915840pp` to `25.333784%`, so continuing the same PPO branch is blocked.

Next action:

Train or score the next candidate with an explicit bull-trend participation floor, SOXLBacktest reference alignment, checkpoint objective loss, and drawdown-drift penalty. A future actor can be retained only after one retained bundle proves fixed-split plus `wf_2020` / `wf_2022` / `wf_2023_2024`, MaxDD below `20%`, return alignment to the current best, and dashboard replay evidence.